In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [26]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# Q1

In [27]:
train.iloc[150]["answer"]
# Therefore according to label it is 2.

'C'

# Q2

In [28]:
row0 = train.iloc[0]
prompt = row0["prompt"]
optB = row0["B"]
formatted_optB = str(prompt)+" [SEP] "+str(optB)
len(formatted_optB)

407

# Q3

In [29]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [43]:
import torch
options = ["A","B","C","D","E"]
input_ids = []
for i in options:
    prompt = row0["prompt"]
    opt = row0[i]
    formatted_opt = str(prompt)+" [SEP] "+str(opt)
    enc = tokenizer(formatted_opt,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
    input_ids.append(enc['input_ids'])
inputs = torch.stack(input_ids)    

In [44]:
reshaped = inputs.permute(1,0,2)
reshaped[:, 0, :]

tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102,  3235,  2002,  5178, 13327,  7164,  2008,  4286,  4839,  2306,
          1037,  2051, 22961,  2008,  2003, 10709,  1998,  2515,  2025,  2031,
          1037,  4225,  2927,  2030,  2203,  1012,  1996,  3276,  2000,  1996,
          2627,  7336, 21894,  2009,  2004,  1037,  3439,  3690,  1010,  1998,
          1996,  3276,  2000,  1996,  2925,  7336,  4526,  1037,  2088,  2008,
          2097, 18094,  3458,  2028,  1005,  1055,  2219,  2051,  1012,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

# Q4

In [47]:
first_16 = train.head(16)
prompts = first_16['prompt'].tolist()

flat_formatted_inputs = []
for idx,row in first_16.iterrows():
    prompt = row['prompt']
    for opt in options:
        formatted = str(prompt)+" [SEP] "+str(row[opt])
        flat_formatted_inputs.append(formatted)

encoded_outputs = tokenizer(
    flat_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

final_input_ids = encoded_outputs['input_ids'].view(16,5,128)

print(f"Total token positions are: {final_input_ids.numel()}")

Total token positions are: 10240


# Q5

In [48]:
from transformers import AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [56]:
model.eval()
options = ['A','B','C','D','E']
row0 = train.iloc[0]
formatted_opts = [str(row0["prompt"])+" [SEP] "+str(row0[opt]) for opt in options]

encoded_inputs = tokenizer(
    formatted_opts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

model_inputs = {
    "input_ids": encoded_inputs["input_ids"].view(1, 5, 128),
    "attention_mask": encoded_inputs["attention_mask"].view(1, 5, 128)
}

with torch.no_grad():
    outputs = model(**model_inputs)

logits = outputs.logits
print(f"Logits shape: {logits.shape}")

Logits shape: torch.Size([1, 5])
